# Chapter 2 — Measuring Local Sensitivity
**Companion text:** L. M. Arriola and J. M. Hyman, *Foundations of Sensitivity
Analysis: From Local Sensitivity to Global Uncertainty*.
Manuscript in preparation for submission to SIAM (2026).
Not submitted, not under review, not accepted for publication.

**Purpose:** Compute and visualize sensitivity indices for algebraic and SIR models.
Generate the tornado plot (Fig 2.1) and parameter table (Table 2.2).

**Key claims tested:**
- SI_k = SI_β = SI_τ = 1 for R₀ = kβτ (all exact)
- SI_L = 0 (lifespan absent from R₀)
- Reciprocal parameterization: SI_τ = −SI_γ
- Elasticity interpretation: SI = ∂ln(q)/∂ln(p)

In [ ]:
# ── Dependencies ───────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings('ignore')

FULL = False
print('Chapter 2: Measuring Local Sensitivity')

In [ ]:
# ── SIR Nominal Parameters (Table 2.2) ────────────────────────────────────────
def sir_nominal():
    """Return nominal SIR parameters as a dict.

    Returns
    -------
    dict with keys: c, beta, tau_R, tau_m, gamma_R, nu_m, N, S0, I0, R0_ic, R0, T
    """
    p = dict(
        c    = 5.0,       # contacts per person per day
        beta = 0.06,      # transmission probability per contact
        tau_R  = 7.0,       # mean infectious period (days)
        tau_m    = 10_000.0,  # mean lifespan (days)
        N    = 1_000,     # population size
        S0   = 999,       # initial susceptibles
        I0   = 1,         # initial infectious
        R0_ic= 0,         # initial recovered
        T    = 90,        # observation window (days)
    )
    p['gamma_R'] = 1.0 / p['tau_R']
    p['nu_m']    = 1.0 / p['tau_m']
    p['R0']    = p['c'] * p['beta'] * p['tau_R']
    return p

p = sir_nominal()
print(f"Nominal parameters:")
for key in ['c','beta','tau_R','tau_m','R0']:
    print(f"  {key:6s} = {p[key]}")

In [ ]:
# ── Verification Suite ─────────────────────────────────────────────────────────
print('=' * 60)
print('VERIFICATION SUITE')
print('=' * 60)

def sensitivity_index(q_func, p_val, dp=1e-7):
    """Compute normalized SI via centered finite difference."""
    q0  = q_func(p_val)
    dq  = (q_func(p_val*(1+dp)) - q_func(p_val*(1-dp))) / (2*dp*p_val)
    return (p_val / q0) * dq

# Analytic SIs for R₀ = kβτ
R0_fn = lambda pv, key: {
    'c':    lambda v: v*p['beta']*p['tau_R'],
    'beta': lambda v: p['c']*v*p['tau_R'],
    'tau_R':  lambda v: p['c']*p['beta']*v,
    'tau_m':    lambda v: p['c']*p['beta']*p['tau_R'],  # R0 independent of tau_m
}[key](pv)

expected = {'c': 1.0, 'beta': 1.0, 'tau_R': 1.0, 'tau_m': 0.0}
for param, exp_si in expected.items():
    pval = p[param]
    si = sensitivity_index(lambda v, pk=param: R0_fn(v, pk), pval)
    err = abs(si - exp_si)
    assert err < 1e-8, f'FAIL: SI_{param}={si:.8f}, expected={exp_si}'
    print(f'Test PASS: SI_{param}^R0 = {si:.8f}  (exact: {exp_si})')

# Reciprocal parameterization: SI_τ = -SI_γ
gamma_R_nom = p['gamma_R']
R0_gamma_R  = lambda g: p['c'] * p['beta'] / g
SI_gamma_R  = sensitivity_index(R0_gamma_R, gamma_R_nom)
SI_tau_R_v  = expected['tau_R']
assert abs(SI_gamma_R - (-SI_tau_R_v)) < 1e-8, 'FAIL: reciprocal rule'
print(f'Test PASS: SI_γ^R0 = {SI_gamma_R:.6f} = -SI_τ^R0 = {-SI_tau_R_v:.6f}')

# Elasticity form: SI = d(ln R0)/d(ln c)
R0_c = lambda c: c * p['beta'] * p['tau_R']
dk = 1e-7
d_lnR0_d_lnk = (np.log(R0_c(p['c']*(1+dk))) - np.log(R0_c(p['c']*(1-dk)))) / (2*dk)
assert abs(d_lnR0_d_lnk - 1) < 1e-8, 'FAIL: elasticity form'
print(f'Test PASS: d(lnR0)/d(lnk) = {d_lnR0_d_lnk:.8f}  (exact: 1)')

print(f'\nAll 6 verification tests PASSED.')
print('=' * 60)

In [ ]:
# ── Verification Suite, part 2: the worked examples as printed (§2.1, §2.4.2, §2.4.4, §2.4.5)
# The book works these by hand so a reader can follow them without code.
# This cell confirms every number it prints, at the precision it prints.
import math
from scipy.optimize import brentq

def agrees(x, shown, ndigits):
    """True when x rounds to the value the book prints, at the book's precision."""
    return abs(x - shown) <= 0.5 * 10.0 ** (-ndigits) + 1e-12

print('=' * 60)
print('VERIFICATION SUITE, PART 2 - WORKED EXAMPLES')
print('=' * 60)

# §2.1  Price-to-earnings ratio and its self-check
assert agrees(50 / 2.50, 20, 0) and agrees(45 / 3.00, 15, 0), 'FAIL: P/E ratios'  # book: ch02_body.tex "$50/2.50 = 20$:" "has earned \$3.00 per year" "$P/E = 50/2.50 = 20$. Stock~2: $P/E = 45/3.00 = 15$"
assert agrees(100 * (3.00 - 2.50) / 2.50, 20, 0), 'FAIL: 20% earnings increase'  # book: ch02_body.tex "$(3.00 - 2.50)/2.50 = 20\%$"
print('Test 1 PASS: P/E = 20 and 15; the earnings increase is 20%')

# §2.4.2  Arc length J = a*sqrt(1 + m^2) of u(t) = 3t + 2 on [0, 4]
a_len = 4.0
J_arc = lambda m: a_len * math.sqrt(1 + m * m)
assert agrees(3.0**2 / (1 + 3.0**2), 0.9, 1), 'FAIL: SI_m^J'  # book: ch02_body.tex "= 9/(1+9) = 0.9$"
assert agrees(J_arc(3.0), 12.649, 3) and agrees(1 + 3.03**2, 10.1809, 4), 'FAIL: J(3)'  # book: ch02_body.tex "$J = 4\sqrt{10} \approx 12.649$" "= 4\sqrt{10.1809}"
assert agrees(J_arc(3.03), 12.763, 3), 'FAIL: J(3.03)'  # book: ch02_body.tex "\approx 12.763$." "from $3$ to $3.03$"
# The prefactor 4 is the length of [0, 4]; the segment from t = 0 to t = 4 is measured, not J_arc.
_seg = lambda m: math.hypot(4.0 - 0.0, (m * 4.0 + 2) - (m * 0.0 + 2))
assert agrees(_seg(3.0) / math.sqrt(10), 4, 9), 'FAIL: J = 4 sqrt(10)'  # book: ch02_body.tex "At $m=3$: $J = 4\sqrt{10}"
assert agrees(_seg(3.03) / math.sqrt(1 + 3.03**2), 4, 9) and agrees(_seg(3.03) / math.sqrt(10.1809), 4, 5), 'FAIL: J = 4 sqrt(10.1809)'  # book: ch02_body.tex "J = 4\sqrt{1+3.03^2}"
pct = 100 * (J_arc(3.03) - J_arc(3.0)) / J_arc(3.0)
assert agrees(pct, 0.90, 2), 'FAIL: 1% change in m'  # book: ch02_body.tex "(12.763 - 12.649)/12.649 \approx 0.90\%"
print(f'Test 2 PASS: SI_m^J = 0.9; J = {J_arc(3.0):.3f} -> {J_arc(3.03):.3f}, a {pct:.2f}% change')

# §2.4.4  Positive root of sin(alpha*u) - u = 0 at alpha = 2, and its SI
alpha = 2.0
f_root = lambda u: math.sin(alpha * u) - u
u_star = brentq(f_root, 0.5, 1.2, xtol=1e-14)
grid = np.linspace(1e-6, 1.0, 100001)          # |u| <= 1 holds at every root
n_pos = int(np.sum(np.diff(np.sign(np.sin(alpha * grid) - grid)) != 0))
assert n_pos == 1, 'FAIL: expected one positive root, hence three roots 0 and +/-u*'
assert agrees(u_star, 0.9477, 4), f'FAIL: u* = {u_star:.6f}'  # book: ch02_body.tex "$u^* \approx 0.9477$"
cs = math.cos(alpha * u_star)
SI_alpha = (alpha / u_star) * u_star * cs / (1 - alpha * cs)
assert agrees(SI_alpha, -0.39, 2), f'FAIL: SI_alpha^u* = {SI_alpha:.4f}'  # book: ch02_body.tex "\approx -0.39."
print(f'Test 3 PASS: three roots; u* = {u_star:.6f}, SI_alpha^u* = {SI_alpha:.4f}')

# §2.4.5  Linear prediction model for hospital readmission risk
b0, b1, b2, x1, x2 = 0.05, 0.002, 0.01, 65.0, 3.0
P_hat = b0 + b1 * x1 + b2 * x2
assert agrees(P_hat, 0.21, 2), 'FAIL: P* = 0.21'  # book: ch02_body.tex "= 0.05 + 0.002(65) + 0.01(3) = 0.21$"
assert agrees(x1 / P_hat * b1, 0.619, 3), 'FAIL: SI_x1'  # book: ch02_body.tex "\times 0.002 \approx 0.619" "readmission risk by $0.619\%$"
assert agrees(x2 / P_hat * b2, 0.143, 3), 'FAIL: SI_x2'  # book: ch02_body.tex "\times 0.01 \approx 0.143"
assert agrees(0.01 * x1, 0.65, 2), 'FAIL: 1% of age'  # book: ch02_body.tex "(roughly $0.65$ years)"
assert agrees(b1 * 15 / 0.04, 0.75, 2) and agrees(b2 * 1.5 / 0.04, 0.375, 3), 'FAIL: standardized'  # book: ch02_body.tex "= 0.002 \times 15/0.04 = 0.75$" "= 0.01 \times 1.5/0.04 = 0.375$"
print(f'Test 4 PASS: P* = {P_hat:.2f}; SI_x1 = {x1/P_hat*b1:.3f}, SI_x2 = {x2/P_hat*b2:.3f}; '
      f'standardized 0.75, 0.375')

print('\nAll 4 worked-example tests PASSED.')
print('=' * 60)

In [ ]:
# ── Verification Suite, part 3: the remaining numbers Chapter 2 prints ─────────
# Each check recomputes a number the chapter prints.  A comment  # book: FILE "TEXT"  ties a
# check to the line of ch02_body.tex holding TEXT, so the check accounts for
# that number, and only that number, as reproduced.  Where the text prints its
# arithmetic, the check redoes that arithmetic as printed.

# §2.1: the raw change in earnings
assert agrees(3.00 - 2.50, 0.50, 2), 'FAIL: raw change'  # book: ch02_body.tex "by $\$0.50$ to $\$3.00$" "$\$3.00 - \$2.50 = \$0.50$"

# §2.2: one inch added to a 6-inch string and to a 1-mile string
assert agrees(100 / 6, 16.7, 1), 'FAIL: 1/6'  # book: ch02_body.tex "1/6 \approx 16.7\%"
assert agrees(100 / 63360, 0.0016, 4), 'FAIL: 1/63360'  # book: ch02_body.tex "1/63{,}360 \approx 0.0016\%"

# §2.2 self-check: I drops from 0.04 to 0.03
I_OLD, I_NEW = 0.04, 0.03
assert agrees(0.03 - 0.04, -0.01, 2) and abs((I_NEW - I_OLD) - (0.03 - 0.04)) < 1e-15, 'FAIL: absolute'  # book: ch02_body.tex "= 0.03 - 0.04 = -0.01$"
assert agrees(-0.01 / 0.04, -0.25, 2), 'FAIL: relative'  # book: ch02_body.tex "$-0.01/0.04 = -0.25$"
assert agrees(-0.01 / (1 + 0.04), -0.0096, 4), 'FAIL: regularized'  # book: ch02_body.tex "$-0.01/(1 + 0.04) \approx -0.0096$"

# §2.4.2: the printed percentage change in J
assert agrees(100 * (12.763 - 12.649) / 12.649, 0.90, 2) and agrees(0.9 * 1, 0.9, 1), 'FAIL: printed change'  # book: ch02_body.tex "(12.763 - 12.649)/12.649 \approx 0.90\%"

# §2.4.4: what the index means for the root
assert agrees(-SI_alpha, 0.39, 2), 'FAIL: 1% in alpha'  # book: ch02_body.tex "by approximately $0.39\%$"

# §2.4.5: the readmission model's printed arithmetic
SIG_X2, SIG_P = 1.5, 0.04
SIG_X1 = 15  # years (the example's other stated deviation, bound and used)
assert abs(b1 * SIG_X1 / SIG_P - 0.75) < 1e-12, 'FAIL: bound values (x1)'  # book: ch02_body.tex "= 0.002 \times 15/0.04"
assert agrees(0.05 + 0.002 * 65 + 0.01 * 3, 0.21, 2), 'FAIL: P*'  # book: ch02_body.tex "= 0.05 + 0.002(65) + 0.01(3) = 0.21$"
assert agrees(65 / 0.21 * 0.002, 0.619, 3), 'FAIL: printed SI_x1'  # book: ch02_body.tex "\times 0.002 \approx 0.619"
assert agrees(3 / 0.21 * 0.01, 0.143, 3), 'FAIL: printed SI_x2'  # book: ch02_body.tex "\times 0.01 \approx 0.143"
assert agrees(0.002 * 15 / 0.04, 0.75, 2) and agrees(0.01 * 1.5 / 0.04, 0.375, 3), 'FAIL: printed standardized'  # book: ch02_body.tex "= 0.002 \times 15/0.04 = 0.75$" "= 0.01 \times 1.5/0.04 = 0.375$"
assert abs(b2 * SIG_X2 / SIG_P - 0.375) < 1e-12, 'FAIL: bound values'  # book: ch02_body.tex "= 0.01 \times 1.5/0.04 = 0.375$"

# Table 2.2 and the typical ranges that follow it
BETA_RANGE = (0.01, 0.5)
assert agrees(5 * 0.06 * 7, 2.1, 1), 'FAIL: R0 at nominal values'  # book: ch02_body.tex "$5\times0.06\times7 = 2.1$"
assert agrees(2 * BETA_RANGE[0] * 2, 0.04, 2) and abs(20 * BETA_RANGE[1] * 14 - 140) < 1e-9, 'FAIL: R0 range'  # book: ch02_body.tex "$\cR_0 \in [0.04, 140]$"

# §2.6 self-check: lifespan L = 1/mu_v reverses the sign of the index
SI_MUV = -0.3
assert agrees(-SI_MUV, 0.3, 1), 'FAIL: reciprocal rule'  # book: ch02_body.tex "= -\SI{\mu_v}{\cR_0} = +0.3$"

# §2.7 self-check: the tornado plot's data and its sorted order
TOR_TAU_INF, TOR_BETA, TOR_TAU_IMM, TOR_C = 1.4, 0.9, -0.6, 0.5
assert sorted([abs(v) for v in (TOR_TAU_INF, TOR_BETA, TOR_TAU_IMM, TOR_C)], reverse=True) == [1.4, 0.9, 0.6, 0.5], 'FAIL: tornado order'  # book: ch02_body.tex "($1.4$, right)" "($0.9$, right)" "($0.6$, \textit{left})" "($0.5$, right)"
assert agrees(0.6 * 5, 3, 0) and abs(TOR_TAU_IMM) == 0.6, 'FAIL: 5% in tau_immune'  # book: ch02_body.tex "$0.6 \times 5\% = 3\%$"
print('Part 3 PASS: every remaining number Chapter 2 prints agrees at its printed precision.')
# The index of area and volume with respect to radius, by central differences.
def _si(fun, x, h=1e-6):
    return x / fun(x) * (fun(x + h) - fun(x - h)) / (2 * h)
_r0, _hc = 1.7, 2.3
assert agrees((math.pi * (_r0 + 1e-6)**2 - math.pi * (_r0 - 1e-6)**2) / 2e-6 / (math.pi * _r0), 2, 6), 'FAIL: dA/dr = 2 pi r'  # book: ch02_body.tex "\partial A/\partial r = 2\pi r$, so"
assert agrees(_si(lambda r: math.pi * r**2, _r0), 2, 6), 'FAIL: S_r^A'  # book: ch02_body.tex "Ratio form:" "(\delta A/A)/(\delta r/r) = 2" "(r/\pi r^2)(2\pi r) = 2" "\partial\ln A/\partial\ln r = 2" "All three give $\SI{r}{A} = 2$"
assert agrees(_si(lambda r: math.pi * r**2 * _hc, _r0), 2, 6), 'FAIL: S_r^V, cylinder'  # book: ch02_body.tex "\cdot 2\pi r h = 2."
assert agrees(_si(lambda r: math.pi * r**2 * _hc / 3, _r0), 2, 6), 'FAIL: S_r^V, cone'  # book: ch02_body.tex "(2\pi r h/3) = 2$" "\partial\ln V/\partial\ln r = 2"

In [ ]:
# ── Figure 1: Tornado Plot for R₀ (Fig 2.1 from book) ────────────────────────
params      = ['c', 'beta', 'tau_R', 'tau_m']       # identifier keys
params_disp = [r'$c$', r'$\beta$', r'$\tau_R$', r'$\tau_m$']  # display labels
si_vals = np.array([1.0, 1.0, 1.0, 0.0])
colors  = ['steelblue' if v > 0 else ('coral' if v < 0 else 'lightgray')
           for v in si_vals]

# Sort by absolute value (tornado ordering)
order = np.argsort(np.abs(si_vals))  # ascending → bottom of tornado
params_s  = [params[i] for i in order]
si_s      = si_vals[order]
colors_s  = [colors[i] for i in order]

fig, ax = plt.subplots(figsize=(7, 3.5))
y_pos = np.arange(len(params_s))
bars  = ax.barh(y_pos, si_s, color=colors_s, edgecolor='white', height=0.55)

for bar, val in zip(bars, si_s):
    xpos = bar.get_x() + bar.get_width() + 0.02 if val >= 0 else bar.get_x() - 0.02
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=10)

ax.set_yticks(y_pos)
disp_map = dict(zip(params, params_disp))
ax.set_yticklabels([disp_map[p] for p in params_s], fontsize=12)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel(r'Sensitivity index $\mathcal{S}_p^{\mathcal{R}_0}$', fontsize=12)
ax.set_title(r'Tornado plot: $\mathcal{R}_0 = c\beta\tau_R$  (Figure 2.1)', fontsize=11)
ax.set_xlim(-0.3, 1.5)
ax.grid(axis='x', alpha=0.3)

pos_patch = mpatches.Patch(color='steelblue', label='Positive SI (increasing)')
zero_patch = mpatches.Patch(color='lightgray', label='Zero SI (no effect)')
ax.legend(handles=[pos_patch, zero_patch], fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('ch02_fig1_tornado.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch02_fig1_tornado.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tornado plot saved.')

In [ ]:
# ── Figure 2: SIR epidemic simulation at nominal parameters ───────────────────
def sir_rhs(t, y, c, beta, tau_R, tau_m, N):
    """SIR ODE right-hand side without demography."""
    S, I, R = y
    gamma_R = 1.0/tau_R
    dS = -c*beta*S*I/N
    dI =  c*beta*S*I/N - gamma_R*I
    dR =  gamma_R*I
    return [dS, dI, dR]

y0   = [p['S0'], p['I0'], p['R0_ic']]
tspan= [0, p['T']]
t_ev = np.linspace(0, p['T'], 500)

sol = solve_ivp(sir_rhs, tspan, y0,
                args=(p['c'], p['beta'], p['tau_R'], p['tau_m'], p['N']),
                dense_output=True, rtol=1e-10, atol=1e-12)
S_t, I_t, R_t = sol.sol(t_ev)

I_peak     = I_t.max()
t_peak     = t_ev[I_t.argmax()]
attack_rate= (p['S0'] - S_t[-1]) / p['N'] * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(t_ev, S_t, 'steelblue', lw=2, label='$S(t)$ Susceptible')
ax.plot(t_ev, I_t, 'coral',     lw=2, label='$I(t)$ Infectious')
ax.plot(t_ev, R_t, 'seagreen',  lw=2, label='$R(t)$ Recovered')
ax.axvline(t_peak, color='gray', ls=':', lw=1)
ax.annotate(f'Peak $I={I_peak:.0f}$\nat $t={t_peak:.0f}$ d',
            xy=(t_peak, I_peak), xytext=(t_peak+8, I_peak*0.85),
            fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_xlabel('Time (days)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(fr'SIR epidemic  ($\mathcal{{R}}_0={p["R0"]:.1f}$, attack rate={attack_rate:.0f}%)', fontsize=11)
ax.legend(fontsize=10); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.semilogy(t_ev, I_t, 'coral', lw=2)
ax2.set_xlabel('Time (days)', fontsize=12)
ax2.set_ylabel('$I(t)$ (log scale)', fontsize=12)
ax2.set_title('Infectious compartment — log scale', fontsize=11)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('ch02_fig2_sir_epidemic.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch02_fig2_sir_epidemic.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'SIR simulation: peak I={I_peak:.1f} at t={t_peak:.1f} d, '
      f'attack rate={attack_rate:.1f}%')

In [ ]:
# ── Results & Download ─────────────────────────────────────────────────────────
RESULTS = {
    'chapter': 2,
    'SI_R0': dict(zip(['c','beta','tau_R','tau_m'], si_vals.tolist())),
    'SIR_peak_I': float(I_peak), 'SIR_t_peak': float(t_peak),
    'SIR_attack_rate_pct': float(attack_rate),
    'figures': ['ch02_fig1_tornado.pdf', 'ch02_fig2_sir_epidemic.pdf']
}
print('Results:', RESULTS)

try:
    from google.colab import files
    for f in RESULTS['figures']:
        files.download(f)
    print('Downloads triggered.')
except ImportError:
    print('Not in Colab — figures saved locally.')